## Toolset

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import cantera as ct
import majordome as mj
import numpy as np

from pathlib import Path
from ruamel.yaml import YAML
from tabulate import tabulate

from IPython.display import Markdown, display

## Setup

In [ ]:
# Total flow rate [Nm³/h].
qdot_fuel = 500.0

# Molar composition of fuel [-].
X_fuel = "CH4: 1"

# Molar composition of oxidizer [-].
X_oxid = "N2: 0.79, O2: 0.21"

# Mechanism for base calculations.
mech = "chemkin/BFER_methane.yaml"
phase = "CH4_BFER_multi"

# Burner diameters [mm].
diam_fuel     = 27
diam_oxid_ext = 342
diam_oxid_int = 76
thick_wall    = 2.5

### Energy input

In [ ]:
normal_flow = mj.NormalFlowRate(mech, X=X_fuel, name=phase)
mdot_fuel = normal_flow(qdot_fuel) / 4

chon = mj.CombustionAtmosphereCHON(mech, phase=phase, basis="mole")
lhv = chon.solution_heating_value(X_fuel, X_oxid)

supply = mj.CombustionPowerSupply(
    power       = lhv * mdot_fuel * 1000,
    equivalence = 1.0,
    fuel        = X_fuel,
    oxidizer    = X_oxid,
    mechanism   = mech,
    phase       = phase,
    basis       = "mole",
)

_ = supply.report(notebook=True)

### Geometry

In [ ]:
A_fuel_pipe = np.pi * (diam_fuel / 2 + thick_wall)**2
A_fuel_one  = np.pi * (diam_fuel / 2)**2
A_fuel_tot  = 4 * A_fuel_pipe

A_air_ext = np.pi * (diam_oxid_ext / 2)**2
A_air_int = np.pi * (diam_oxid_int / 2)**2

A_air = A_air_ext - A_air_int - A_fuel_tot
A_air_one = A_air / 4

e_wall = float(0.001 * thick_wall)
D_fuel = float(0.001 * diam_fuel)
D_oxid = float(0.001 * np.sqrt(4 * (A_air_one + A_fuel_pipe) / np.pi))

In [ ]:
geometry_data = {
    "wedge_angle": 2.0,
    "len_inlet_ax": 3 * D_fuel,
    "len_inlet_an": 3 * D_fuel,
    "len_disperse": 1.00,
    "len_flue_out": 2.00,
    "dia_inlet_ax": D_fuel,
    "thk_inlet_wl": e_wall,
    "thk_inlet_an": D_oxid / 2,
    "thk_side_box": 2 * D_oxid,
}

yaml = YAML()
yaml.default_flow_style = False
yaml.indent(mapping=2, sequence=4, offset=2)

output_file = Path("domain.yaml")
with output_file.open("w", encoding="utf-8") as f:
    yaml.dump(geometry_data, f)

### Kinetics data

In [ ]:
sutherland = mj.SutherlandFitting(mech, name=phase)
sutherland.fit(np.linspace(400, 2300, 100))

transport = Path(mech).parent / "transport"
transport.unlink()

_ = sutherland.as_openfoam_dict(transport)

In [ ]:
#| echo: false
# Use the following for inspection:
# figs = Path(mech).parent / "media"
# figs.mkdir(exist_ok=True)

# for s in coef["species"]:
#     p = sutherland.plot_species(s)
#     p.savefig(figs / f"{s}.png")

In [ ]:
#| echo: false
#| tbl-cap: Sutherland fitting coefficients for transport properties.
#| label: tbl-sutherland-coefs
sutherland.coefs_table

### Conditions setup

In [ ]:
def mean_velocity(mdot, rho, T, A):
    """ Temperature corrected mean flow velocity. """
    return mdot / (rho * A)


def characteristic_length(A, f=0.07):
    """ Characteristic length for turbulence. """
    return f * np.sqrt(4.0 * A / np.pi)


def turbulent_kinetic_energy(U, I=0.05):
    """ Computes the turbulent kinetic energy. """
    return 1.5 * (U * I)**2


def turbulent_dissipation_rate(k, L, C_mu=0.09):
    """ Computes the turbulent dissipation rate. """
    return (C_mu**0.75) * (k**1.5) / L

In [ ]:
setup_dict = mj.FoamDictFile(path="0.orig/_shared")


def flow_workflow(name, mdot, rho0, T, A, I=0.05):
    rho = (273.15 / T) * rho0

    U = mean_velocity(mdot, rho, T, A)
    L = characteristic_length(A)
    k = turbulent_kinetic_energy(U, I)
    e = turbulent_dissipation_rate(k, L)

    data = [
        ("Temperature", "K", T),
        ("Mean velocity", "m/s", U),
        ("Characteristic length", "mm", 1000*L),
        ("Turbulent kinetic energy", "m²/s²", k),
        ("Turbulent dissipation rate", "m²/s³", e),
    ]
    headers = ["Quantity", "Unit", "Value"]
    display(Markdown(tabulate(data, headers=headers, tablefmt="github")))

    key_name = name.upper()
    setup_dict.set(f"{key_name}_T", T)
    setup_dict.set(f"{key_name}_U", U)
    setup_dict.set(f"{key_name}_RHO", rho)
    setup_dict.set(f"{key_name}_L", L)
    setup_dict.set(f"{key_name}_I", I)
    setup_dict.set(f"{key_name}_K", k)
    setup_dict.set(f"{key_name}_EPSILON", e)
    setup_dict.save()


In [ ]:
flow_workflow(
    name = "FUEL",
    mdot = supply.fuel_mass,
    rho0 = supply.fuel_normal_density,
    T    = 298.15,
    A    = 1.0e-06 * A_fuel_one
)

In [ ]:
flow_workflow(
    name = "OXID",
    mdot = supply.oxidizer_mass,
    rho0 = supply.oxidizer_normal_density,
    T    = 298.15,
    A    = 1.0e-06 * A_air_one
)